#  Análisis de Datos — Total Automotriz La Victoria

**Datos:** Sep 2023 → Sep 2025 (2 años de operación)  
**Objetivo:** Entender qué productos vender, cuáles rotan más, márgenes y tendencias  
**Herramientas:** pandas, numpy, matplotlib, seaborn

---

In [4]:

# ══════════════════════════════════════════════════
# Estas son las librerías son las que vamos a utilizar.

import pandas as pd          # Manejo de datos (tablas/DataFrames)
import numpy as np           # Cálculos matemáticos y estadísticas
import matplotlib.pyplot as plt  # Gráficas base
import seaborn as sns        # Gráficas más bonitas sobre matplotlib
import warnings
warnings.filterwarnings('ignore')

# Configuración visual
plt.rcParams['figure.figsize'] = (14, 5)
sns.set_theme(style="whitegrid")

# Colores del negocio
COLOR_NAVY = '#1B2A4A'
COLOR_GOLD = '#D4AF37'

print(" Librerías cargadas correctamente")
print(f"   pandas  versión: {pd.__version__}")
print(f"   numpy   versión: {np.__version__}")

 Librerías cargadas correctamente
   pandas  versión: 2.2.3
   numpy   versión: 2.1.3


In [5]:
# ══════════════════════════════════════════════════
# CELDA 3 — CARGAR EL ARCHIVO EXCEL
# ══════════════════════════════════════════════════

ARCHIVO = r"F:\ANALISIS DATOS TOTAL AUTOMOTRIZ.xlsx"

# Cargar cada hoja en su propio DataFrame
df_facturas      = pd.read_excel(ARCHIVO, sheet_name='FACTURAS 20-09-2023-20-09-2025')
df_articulos     = pd.read_excel(ARCHIVO, sheet_name='ARTICULOS ')
df_clientes      = pd.read_excel(ARCHIVO, sheet_name='CLIENTES')
df_departamentos = pd.read_excel(ARCHIVO, sheet_name='DEPARTAMENTOS')
df_proveedores   = pd.read_excel(ARCHIVO, sheet_name='PROVEEDORES')

print(" Archivo cargado exitosamente!\n")
print(f"{'Hoja':<30} {'Filas':>8}  {'Columnas':>9}")
print("-" * 50)
for nombre, df in [
    ('FACTURAS',      df_facturas),
    ('ARTICULOS',     df_articulos),
    ('CLIENTES',      df_clientes),
    ('DEPARTAMENTOS', df_departamentos),
    ('PROVEEDORES',   df_proveedores),
]:
    print(f"{nombre:<30} {len(df):>8,}  {len(df.columns):>9}")

 Archivo cargado exitosamente!

Hoja                              Filas   Columnas
--------------------------------------------------
FACTURAS                         21,928          7
ARTICULOS                        10,120         14
CLIENTES                            735         13
DEPARTAMENTOS                        37          2
PROVEEDORES                         282         13


In [7]:
# ══ — EXPLORACIÓN INICIAL
# Lo PRIMERO en cualquier proyecto de datos:
# entender QUÉ tenés antes de analizar.
# ══════════════════════════════════════════════════

print("=" * 55)
print("  EXPLORANDO: FACTURAS")
print("=" * 55)

# .shape → filas y columnas
print(f"\n Tamaño: {df_facturas.shape[0]:,} filas × {df_facturas.shape[1]} columnas")

# .columns → nombres de columnas
print(f"\n Columnas:\n  {df_facturas.columns.tolist()}")

# .head() → primeras filas (siempre al final de la celda
#            para que Jupyter lo muestre bonito)
print("\n Primeras 5 filas:")
df_facturas.head()

  EXPLORANDO: FACTURAS

 Tamaño: 21,928 filas × 7 columnas

 Columnas:
  ['No.factura', 'Nombre cliente', 'Fecha', 'Tipo factura', 'Saldo capital', 'Vendedor', 'Abono']

 Primeras 5 filas:


,No.factura,Nombre cliente,Fecha,Tipo factura,Saldo capital,Vendedor,Abono
0,40497,jose,2025-09-20,Contado,3500.0,Andrew Chamorro,0.0
1,2310-B,OSCAR,2025-09-20,Contado,1600.0,Alex Chamorro,0.0
2,2309-B,ROY,2025-09-20,Contado,1200.0,Alex Chamorro,0.0
3,40496,ROY,2025-09-20,Contado,4700.0,Alex Chamorro,0.0
4,40495,ROY,2025-09-20,Contado,0.0,Charlie Marenco Chamorro,0.0


In [8]:
# ═════════════════— TIPOS DE DATOS Y VALORES NULOS
# ══════════════════════════════════════════════════
# Antes de analizar CUALQUIER dato, hay que verificar:
# 1. ¿Cada columna tiene el tipo correcto?
# 2. ¿Hay datos faltantes (nulos)?

print(" TIPOS DE DATOS ACTUALES:")
print("-" * 35)
print(df_facturas.dtypes)

print("\n\n  VALORES NULOS POR COLUMNA:")
print("-" * 35)
nulos = df_facturas.isnull().sum()
pct   = (nulos / len(df_facturas) * 100).round(1)
resumen = pd.DataFrame({
    'Nulos':        nulos,
    'Porcentaje %': pct
})
print(resumen)

print("\n\n INTERPRETACIÓN:")
print(f"   Columnas sin ningún nulo:    {(nulos == 0).sum()}")
print(f"   Columnas CON algún nulo:     {(nulos > 0).sum()}")
print(f"\n     'Saldo capital' tiene tipo: {df_facturas['Saldo capital'].dtype}")
print(f"     'Fecha' tiene tipo:          {df_facturas['Fecha'].dtype}")
print("\n   → Si Fecha no dice datetime64, hay que convertirla")
print("   → Si Saldo capital no dice float64, hay que convertirlo")
print("   → Eso lo hacemos en la siguiente celda: LIMPIEZA")

 TIPOS DE DATOS ACTUALES:
-----------------------------------
No.factura                object
Nombre cliente            object
Fecha             datetime64[ns]
Tipo factura              object
Saldo capital            float64
Vendedor                  object
Abono                    float64
dtype: object


  VALORES NULOS POR COLUMNA:
-----------------------------------
                Nulos  Porcentaje %
No.factura          1           0.0
Nombre cliente      1           0.0
Fecha               1           0.0
Tipo factura        1           0.0
Saldo capital       1           0.0
Vendedor          880           4.0
Abono               1           0.0


 INTERPRETACIÓN:
   Columnas sin ningún nulo:    0
   Columnas CON algún nulo:     7

     'Saldo capital' tiene tipo: float64
     'Fecha' tiene tipo:          datetime64[ns]

   → Si Fecha no dice datetime64, hay que convertirla
   → Si Saldo capital no dice float64, hay que convertirlo
   → Eso lo hacemos en la siguiente celda: LIM

In [9]:
# ════════LIMPIEZA DE DATOS
# Antes de analizar, los datos tienen que estar
# en el formato correcto. Esto se llama "cleaning".
# ══════

# Renombrar columnas para trabajar más fácil
df_facturas.columns = ['factura_id', 'cliente', 'fecha', 
                        'tipo', 'total', 'vendedor', 'abono']

# Convertir fecha al tipo correcto (datetime)
df_facturas['fecha'] = pd.to_datetime(df_facturas['fecha'], errors='coerce')

# Convertir montos a número
df_facturas['total'] = pd.to_numeric(df_facturas['total'], errors='coerce')
df_facturas['abono'] = pd.to_numeric(df_facturas['abono'], errors='coerce').fillna(0)

# Crear columnas nuevas derivadas de la fecha
# Esto es MUY común en análisis de ventas
df_facturas['año']       = df_facturas['fecha'].dt.year
df_facturas['mes']       = df_facturas['fecha'].dt.month
df_facturas['mes_nombre']= df_facturas['fecha'].dt.strftime('%b')
df_facturas['dia_semana']= df_facturas['fecha'].dt.day_name()
df_facturas['trimestre'] = df_facturas['fecha'].dt.quarter

# Limpiar texto
df_facturas['cliente']  = df_facturas['cliente'].astype(str).str.upper().str.strip()
df_facturas['vendedor'] = df_facturas['vendedor'].astype(str).str.strip()

# Eliminar filas inválidas
antes = len(df_facturas)
df_facturas = df_facturas.dropna(subset=['fecha', 'total'])
df_facturas = df_facturas[df_facturas['total'] > 0]
despues = len(df_facturas)

print(" LIMPIEZA COMPLETADA")
print(f"   Filas antes:   {antes:,}")
print(f"   Filas después: {despues:,}")
print(f"   Eliminadas:    {antes - despues:,} (inválidas)")
print(f"\n Período de datos:")
print(f"   Desde: {df_facturas['fecha'].min().strftime('%d/%m/%Y')}")
print(f"   Hasta: {df_facturas['fecha'].max().strftime('%d/%m/%Y')}")
print(f"\n Vista previa limpia:")
df_facturas.head(3)

 LIMPIEZA COMPLETADA
   Filas antes:   21,928
   Filas después: 17,165
   Eliminadas:    4,763 (inválidas)

 Período de datos:
   Desde: 20/09/2023
   Hasta: 20/09/2025

 Vista previa limpia:


,factura_id,cliente,fecha,tipo,total,vendedor,abono,año,mes,mes_nombre,dia_semana,trimestre
0,40497,JOSE,2025-09-20,Contado,3500.0,Andrew Chamorro,0.0,2025.0,9.0,Sep,Saturday,3.0
1,2310-B,OSCAR,2025-09-20,Contado,1600.0,Alex Chamorro,0.0,2025.0,9.0,Sep,Saturday,3.0
2,2309-B,ROY,2025-09-20,Contado,1200.0,Alex Chamorro,0.0,2025.0,9.0,Sep,Saturday,3.0


In [10]:
# ══════════════════════════════════════════════════
# CELDA 7 — ESTADÍSTICAS DESCRIPTIVAS
# ══════════════════════════════════════════════════
# .describe() es el comando más poderoso para
# entender rápidamente cualquier conjunto de datos.
# Te da 8 estadísticas de golpe.
# ══════════════════════════════════════════════════

print(" ESTADÍSTICAS DESCRIPTIVAS — MONTO DE VENTAS (₡)")
print("=" * 50)

desc = df_facturas['total'].describe()

print(f"\n  count  = {int(desc['count']):>10,}  ← cantidad de transacciones")
print(f"  mean   = ₡{desc['mean']:>10,.0f}  ← ticket PROMEDIO")
print(f"  std    = ₡{desc['std']:>10,.0f}  ← qué tan dispersos son los montos")
print(f"  min    = ₡{desc['min']:>10,.0f}  ← venta más pequeña")
print(f"  25%    = ₡{desc['25%']:>10,.0f}  ← 25% de ventas están BAJO este monto")
print(f"  50%    = ₡{desc['50%']:>10,.0f}  ← venta MEDIANA (el centro real)")
print(f"  75%    = ₡{desc['75%']:>10,.0f}  ← 75% de ventas están BAJO este monto")
print(f"  max    = ₡{desc['max']:>10,.0f}  ← venta más grande registrada")

print("\n" + "=" * 50)
print("  RESUMEN EJECUTIVO")
print("=" * 50)
total_ventas = df_facturas['total'].sum()
dias_total   = (df_facturas['fecha'].max() - df_facturas['fecha'].min()).days

print(f"\n   Ventas TOTALES (2 años):  ₡{total_ventas:>12,.0f}")
print(f"   Promedio MENSUAL:          ₡{total_ventas/(dias_total/30):>12,.0f}")
print(f"   Promedio DIARIO:           ₡{total_ventas/dias_total:>12,.0f}")
print(f"   Total transacciones:        {len(df_facturas):>12,}")

print("\n\n RESUMEN POR AÑO:")
print("-" * 60)
por_año = df_facturas.groupby('año').agg(
    facturas        = ('total', 'count'),
    ventas_total    = ('total', 'sum'),
    ticket_promedio = ('total', 'mean'),
    venta_maxima    = ('total', 'max')
).reset_index()

for _, row in por_año.iterrows():
    print(f"\n   AÑO {int(row['año'])}:")
    print(f"     Facturas emitidas:   {int(row['facturas']):>8,}")
    print(f"     Ventas totales:      ₡{row['ventas_total']:>10,.0f}")
    print(f"     Ticket promedio:     ₡{row['ticket_promedio']:>10,.0f}")
    print(f"     Venta más alta:      ₡{row['venta_maxima']:>10,.0f}")

 ESTADÍSTICAS DESCRIPTIVAS — MONTO DE VENTAS (₡)

  count  =     17,165  ← cantidad de transacciones
  mean   = ₡    11,198  ← ticket PROMEDIO
  std    = ₡    22,671  ← qué tan dispersos son los montos
  min    = ₡         2  ← venta más pequeña
  25%    = ₡     2,850  ← 25% de ventas están BAJO este monto
  50%    = ₡     5,400  ← venta MEDIANA (el centro real)
  75%    = ₡    11,000  ← 75% de ventas están BAJO este monto
  max    = ₡ 1,050,947  ← venta más grande registrada

  RESUMEN EJECUTIVO

   Ventas TOTALES (2 años):  ₡ 192,218,919
   Promedio MENSUAL:          ₡   7,888,601
   Promedio DIARIO:           ₡     262,953
   Total transacciones:              17,165


 RESUMEN POR AÑO:
------------------------------------------------------------

   AÑO 2023:
     Facturas emitidas:      1,986
     Ventas totales:      ₡25,152,096
     Ticket promedio:     ₡    12,665
     Venta más alta:      ₡   474,043

   AÑO 2024:
     Facturas emitidas:      8,479
     Ventas totales:      ₡93

In [11]:
# ══════════════════════════════════════════════════
# CELDA 7B — ¿CUÁLES FUERON LAS VENTAS MÁS ALTAS?
# ══════════════════════════════════════════════════
# .nlargest(n, columna) → trae las N filas más grandes
# .loc[] → filtra filas según una condición

print(" VENTA MÁS ALTA DE CADA AÑO")
print("=" * 60)

for año in [2023, 2024, 2025]:
    # Filtrar solo ese año
    df_año = df_facturas[df_facturas['año'] == año]
    
    # Tomar las 3 ventas más altas de ese año
    top3 = df_año.nlargest(3, 'total')[['factura_id','fecha','cliente','vendedor','total','tipo']]
    
    print(f"\n   AÑO {año} — Top 3 ventas más altas:")
    print(f"  {'-'*55}")
    for i, (_, row) in enumerate(top3.iterrows(), 1):
        print(f"  {i}. Factura:  {row['factura_id']}")
        print(f"     Fecha:     {row['fecha'].strftime('%d/%m/%Y')}")
        print(f"     Cliente:   {row['cliente']}")
        print(f"     Vendedor:  {row['vendedor']}")
        print(f"     Monto:     ₡{row['total']:,.0f}")
        print(f"     Tipo:      {row['tipo']}")
        print()

 VENTA MÁS ALTA DE CADA AÑO

   AÑO 2023 — Top 3 ventas más altas:
  -------------------------------------------------------
  1. Factura:  22345
     Fecha:     15/11/2023
     Cliente:   DAGOBERTO MEJIAS GOMEZ LO DE RAQUEL
     Vendedor:  Andrew Chamorro
     Monto:     ₡474,043
     Tipo:      Contado

  2. Factura:  22498
     Fecha:     22/11/2023
     Cliente:   ASOCIACION DE DESARROLLO INTEGRAL DE FINCA SAN JOSE Y CHIRRIPO DE RIO FRIO HORQU
     Vendedor:  Jenny Trejos Perez
     Monto:     ₡319,688
     Tipo:      Contado

  3. Factura:  22306
     Fecha:     14/11/2023
     Cliente:   JOSE ANGEL FLORES VALVERDE
     Vendedor:  Charlie Marenco Chamorro
     Monto:     ₡316,620
     Tipo:      Contado


   AÑO 2024 — Top 3 ventas más altas:
  -------------------------------------------------------
  1. Factura:  26672
     Fecha:     19/04/2024
     Cliente:   JOSE DAVID MORALES ARAYA
     Vendedor:  Charlie Marenco Chamorro
     Monto:     ₡665,570
     Tipo:      Contado

  2.

In [15]:
# CELDA 8 — ANÁLISIS POR VENDEDOR

por_vendedor = df_facturas.groupby('vendedor').agg(
    facturas        = ('total', 'count'),
    ventas_total    = ('total', 'sum'),
    ticket_promedio = ('total', 'mean'),
    venta_maxima    = ('total', 'max'),
    venta_minima    = ('total', 'min'),
).reset_index()

total_general = por_vendedor['ventas_total'].sum()
por_vendedor['participacion_%'] = (por_vendedor['ventas_total'] / total_general * 100).round(1)
por_vendedor = por_vendedor.sort_values('ventas_total', ascending=False).reset_index(drop=True)

print(" ANÁLISIS COMPLETO POR VENDEDOR")
print("=" * 65)
print(f"\n{'#':<3} {'Vendedor':<25} {'Facturas':>8} {'Ventas Total':>14} {'Participación':>14}")
print("-" * 70)

for i, row in por_vendedor.iterrows():
    barra = '█' * int(row['participacion_%'] / 2)
    print(f"{i+1:<3} {row['vendedor']:<25} {int(row['facturas']):>8,} "
          f"₡{row['ventas_total']:>12,.0f} {row['participacion_%']:>12.1f}%  {barra}")
    print(f"    {'':25} Ticket prom: ₡{row['ticket_promedio']:>10,.0f}   "
          f"Máx: ₡{row['venta_maxima']:>8,.0f}")
    print()

 ANÁLISIS COMPLETO POR VENDEDOR

#   Vendedor                  Facturas   Ventas Total  Participación
----------------------------------------------------------------------
1   Charlie Marenco Chamorro     6,914 ₡  86,563,457         45.0%  ██████████████████████
                              Ticket prom: ₡    12,520   Máx: ₡ 665,570

2   Alex Chamorro                4,806 ₡  62,398,545         32.5%  ████████████████
                              Ticket prom: ₡    12,983   Máx: ₡1,050,947

3   Vladimir Hernandez Trejos    2,845 ₡  17,673,712          9.2%  ████
                              Ticket prom: ₡     6,212   Máx: ₡  83,700

4   Jenny Trejos Perez             969 ₡  10,627,685          5.5%  ██
                              Ticket prom: ₡    10,968   Máx: ₡ 619,289

5   nan                            422 ₡   5,788,044          3.0%  █
                              Ticket prom: ₡    13,716   Máx: ₡ 179,350

6   Andrew Chamorro                632 ₡   5,449,266          2.8%  █
 

In [16]:
# CELDA 9 — EVOLUCIÓN DE VENDEDORES POR AÑO

print(" VENTAS POR VENDEDOR — AÑO A AÑO")
print("=" * 65)

df_vendedores = df_facturas[~df_facturas['vendedor'].isin(['SIN ASIGNAR', 'nan'])]

pivot = df_vendedores.pivot_table(
    index    = 'vendedor',
    columns  = 'año',
    values   = 'total',
    aggfunc  = 'sum',
    fill_value = 0
).round(0)

pivot['TOTAL'] = pivot.sum(axis=1)
pivot = pivot.sort_values('TOTAL', ascending=False)

print(f"\n  {'Vendedor':<25}", end='')
for col in pivot.columns:
    print(f"  {str(col):>14}", end='')
print()
print("  " + "-" * 75)

for vendedor, row in pivot.iterrows():
    print(f"  {vendedor:<25}", end='')
    for val in row:
        print(f"  ₡{val:>12,.0f}", end='')
    print()

print(f"\n\n CRECIMIENTO POR VENDEDOR (2024 → 2025):")
print("-" * 50)
for vendedor, row in pivot.iterrows():
    v2024 = row.get(2024, 0)
    v2025 = row.get(2025, 0)
    if v2024 > 0:
        crecimiento = ((v2025 - v2024) / v2024) * 100
        emoji = "" if crecimiento >= 0 else "📉"
        print(f"  {emoji} {vendedor:<28} {crecimiento:>+.1f}%")

 VENTAS POR VENDEDOR — AÑO A AÑO

  Vendedor                           2023.0          2024.0          2025.0           TOTAL
  ---------------------------------------------------------------------------
  Charlie Marenco Chamorro   ₡  10,446,665  ₡  47,583,011  ₡  28,533,781  ₡  86,563,457
  Alex Chamorro              ₡  10,016,143  ₡  28,414,705  ₡  23,967,697  ₡  62,398,545
  Vladimir Hernandez Trejos  ₡   1,741,555  ₡   8,454,561  ₡   7,477,596  ₡  17,673,712
  Jenny Trejos Perez         ₡   1,576,708  ₡   5,172,896  ₡   3,878,081  ₡  10,627,685
  Andrew Chamorro            ₡   1,263,227  ₡   1,656,302  ₡   2,529,737  ₡   5,449,266
  LUIS MARENCO CHAMORRO      ₡           0  ₡           0  ₡   1,549,590  ₡   1,549,590
  Anderson Rojas Gonzales    ₡           0  ₡           0  ₡   1,173,195  ₡   1,173,195
  Tamara Chinchilla          ₡           0  ₡     910,652  ₡       3,700  ₡     914,352
  Pablo Miguel Rojas Chamorro  ₡           0  ₡           0  ₡      81,073  ₡      81,073




In [17]:
# ════════════════════════════════════════════════— ¿POR QUÉ CAYERON LAS VENTAS EN 2025?
# Investigación mes a mes
# ══════════════════════════════════════════════════

print(" INVESTIGANDO LA CAÍDA DE VENTAS 2024 → 2025")
print("=" * 60)

# DATO BASE
v2024 = df_facturas[df_facturas['año'] == 2024]['total'].sum()
v2025 = df_facturas[df_facturas['año'] == 2025]['total'].sum()
diferencia = v2025 - v2024
caida_pct  = (diferencia / v2024) * 100

print(f"\n  Ventas 2024:     ₡{v2024:>12,.0f}")
print(f"  Ventas 2025:     ₡{v2025:>12,.0f}")
print(f"  Diferencia:      ₡{diferencia:>12,.0f}")
print(f"  Variación:        {caida_pct:>+.1f}%")

# ── PISTA 1 — ¿Cuántos meses tenemos de cada año? ──
print(f"\n\n PISTA 1 — ¿Cuántos meses de datos hay por año?")
print("-" * 50)
meses_por_año = df_facturas.groupby('año')['mes'].nunique()
facturas_por_año = df_facturas.groupby('año')['total'].count()

for año in [2023, 2024, 2025]:
    meses = meses_por_año.get(año, 0)
    facts = facturas_por_año.get(año, 0)
    print(f"  {año}: {meses} meses de datos  →  {facts:,} facturas")

# ── PISTA 2 — Ventas MES A MES comparando 2024 vs 2025 ──
print(f"\n\n PISTA 2 — VENTAS MES A MES (2024 vs 2025)")
print("-" * 60)

meses_nombre = {1:'Enero',2:'Febrero',3:'Marzo',4:'Abril',
                5:'Mayo',6:'Junio',7:'Julio',8:'Agosto',
                9:'Septiembre',10:'Octubre',11:'Noviembre',12:'Diciembre'}

ventas_mes = df_facturas.groupby(['año','mes'])['total'].sum().reset_index()

print(f"\n  {'Mes':<12} {'2024':>14} {'2025':>14} {'Diferencia':>14} {'Var%':>8}")
print(f"  {'-'*58}")

for mes in range(1, 13):
    v_2024 = ventas_mes[(ventas_mes['año']==2024) & (ventas_mes['mes']==mes)]['total'].sum()
    v_2025 = ventas_mes[(ventas_mes['año']==2025) & (ventas_mes['mes']==mes)]['total'].sum()
    
    if v_2024 == 0 and v_2025 == 0:
        continue
        
    diff = v_2025 - v_2024
    if v_2024 > 0:
        var = (diff / v_2024) * 100
        emoji = "" if var >= 0 else ""
        print(f"  {meses_nombre[mes]:<12} ₡{v_2024:>12,.0f} ₡{v_2025:>12,.0f} "
              f"₡{diff:>12,.0f} {var:>+7.1f}% {emoji}")
    else:
        print(f"  {meses_nombre[mes]:<12} {'(sin datos)':>14} ₡{v_2025:>12,.0f} {'---':>14}")

 INVESTIGANDO LA CAÍDA DE VENTAS 2024 → 2025

  Ventas 2024:     ₡  93,950,753
  Ventas 2025:     ₡  73,116,070
  Diferencia:      ₡ -20,834,683
  Variación:        -22.2%


 PISTA 1 — ¿Cuántos meses de datos hay por año?
--------------------------------------------------
  2023: 4 meses de datos  →  1,986 facturas
  2024: 12 meses de datos  →  8,479 facturas
  2025: 9 meses de datos  →  6,700 facturas


 PISTA 2 — VENTAS MES A MES (2024 vs 2025)
------------------------------------------------------------

  Mes                    2024           2025     Diferencia     Var%
  ----------------------------------------------------------
  Enero        ₡   7,410,372 ₡   9,828,130 ₡   2,417,758   +32.6% 
  Febrero      ₡   6,527,940 ₡   9,889,646 ₡   3,361,706   +51.5% 
  Marzo        ₡   8,550,041 ₡   7,316,790 ₡  -1,233,251   -14.4% 
  Abril        ₡   7,579,428 ₡   8,545,993 ₡     966,565   +12.8% 
  Mayo         ₡   8,609,028 ₡   7,403,975 ₡  -1,205,052   -14.0% 
  Junio        ₡   8,0

In [18]:
# ═════════════════════════════════════════════ — ¿FUE MENOS CLIENTES O MENOS GASTO?
# Hay DOS razones por las que pueden caer las ventas:
# 1. Vinieron MENOS clientes (menos facturas)
# 2. Cada cliente gastó MENOS (ticket más bajo)
# ══════════════════════════════════════════════════

print(" PISTA 3 — ¿MENOS CLIENTES O MENOS GASTO POR CLIENTE?")
print("=" * 60)

resumen = df_facturas.groupby('año').agg(
    facturas        = ('total', 'count'),
    ventas_total    = ('total', 'sum'),
    ticket_promedio = ('total', 'mean'),
    clientes_unicos = ('cliente', 'nunique')
).reset_index()

for _, row in resumen.iterrows():
    print(f"\n   AÑO {int(row['año'])}:")
    print(f"     Facturas emitidas:    {int(row['facturas']):>8,}")
    print(f"     Clientes únicos:      {int(row['clientes_unicos']):>8,}")
    print(f"     Ventas totales:       ₡{row['ventas_total']:>10,.0f}")
    print(f"     Ticket promedio:      ₡{row['ticket_promedio']:>10,.0f}")

# Comparar 2024 vs 2025
f2024 = resumen[resumen['año']==2024]['facturas'].values[0]
f2025 = resumen[resumen['año']==2025]['facturas'].values[0]
t2024 = resumen[resumen['año']==2024]['ticket_promedio'].values[0]
t2025 = resumen[resumen['año']==2025]['ticket_promedio'].values[0]
c2024 = resumen[resumen['año']==2024]['clientes_unicos'].values[0]
c2025 = resumen[resumen['año']==2025]['clientes_unicos'].values[0]

print(f"\n\n   COMPARACIÓN DIRECTA 2024 → 2025:")
print(f"  {'':30} {'2024':>10} {'2025':>10} {'Cambio':>10}")
print(f"  {'-'*60}")
print(f"  {'Facturas emitidas':<30} {int(f2024):>10,} {int(f2025):>10,} "
      f"{((f2025-f2024)/f2024*100):>+9.1f}%")
print(f"  {'Clientes únicos':<30} {int(c2024):>10,} {int(c2025):>10,} "
      f"{((c2025-c2024)/c2024*100):>+9.1f}%")
print(f"  {'Ticket promedio':<30} ₡{t2024:>9,.0f} ₡{t2025:>9,.0f} "
      f"{((t2025-t2024)/t2024*100):>+9.1f}%")

print(f"\n\n   DIAGNÓSTICO:")
if f2025 < f2024 and t2025 < t2024:
    print("   DOBLE PROBLEMA: Menos clientes Y menor gasto por cliente")
elif f2025 < f2024:
    print("    El problema es VOLUMEN: vinieron menos clientes")
elif t2025 < t2024:
    print("    El problema es TICKET: los clientes gastaron menos")

 PISTA 3 — ¿MENOS CLIENTES O MENOS GASTO POR CLIENTE?

   AÑO 2023:
     Facturas emitidas:       1,986
     Clientes únicos:         1,004
     Ventas totales:       ₡25,152,096
     Ticket promedio:      ₡    12,665

   AÑO 2024:
     Facturas emitidas:       8,479
     Clientes únicos:         3,077
     Ventas totales:       ₡93,950,753
     Ticket promedio:      ₡    11,080

   AÑO 2025:
     Facturas emitidas:       6,700
     Clientes únicos:         2,487
     Ventas totales:       ₡73,116,070
     Ticket promedio:      ₡    10,913


   COMPARACIÓN DIRECTA 2024 → 2025:
                                       2024       2025     Cambio
  ------------------------------------------------------------
  Facturas emitidas                   8,479      6,700     -21.0%
  Clientes únicos                     3,077      2,487     -19.2%
  Ticket promedio                ₡   11,080 ₡   10,913      -1.5%


   DIAGNÓSTICO:
   DOBLE PROBLEMA: Menos clientes Y menor gasto por cliente


In [19]:
# ══════════════════════════════════════════════════— IDENTIFICAR EL MES EXACTO DE LA CAÍDA
# ══════════════════════════════════════════════════

print(" PISTA 4 — ¿EN QUÉ MES EMPEZÓ LA CAÍDA?")
print("=" * 55)

ventas_mensual = df_facturas.groupby(['año','mes']).agg(
    total    = ('total', 'sum'),
    facturas = ('total', 'count')
).reset_index()
ventas_mensual['periodo'] = ventas_mensual.apply(
    lambda r: f"{meses_nombre[int(r['mes'])][:3]} {int(r['año'])}", axis=1)

# Ordenar cronológicamente
ventas_mensual = ventas_mensual.sort_values(['año','mes'])

# Calcular crecimiento mes a mes
ventas_mensual['var_mes_%'] = ventas_mensual['total'].pct_change() * 100

print(f"\n  {'Período':<12} {'Ventas':>14} {'Facturas':>10} {'Var vs mes anterior':>20}")
print(f"  {'-'*60}")
for _, row in ventas_mensual.iterrows():
    var_str = f"{row['var_mes_%']:>+.1f}%" if not pd.isna(row['var_mes_%']) else "  base"
    emoji = "" if row.get('var_mes_%', 0) >= 0 else ""
    separador = "  ──────" if row['mes'] == 1 else ""
    print(f"  {row['periodo']:<12} ₡{row['total']:>12,.0f} {int(row['facturas']):>10,}"
          f"  {var_str:>10} {emoji} {separador}")

 PISTA 4 — ¿EN QUÉ MES EMPEZÓ LA CAÍDA?

  Período              Ventas   Facturas  Var vs mes anterior
  ------------------------------------------------------------
  Sep 2023     ₡   2,952,793        220        base  
  Oct 2023     ₡   7,896,362        584     +167.4%  
  Nov 2023     ₡   6,702,330        529      -15.1%  
  Dic 2023     ₡   7,600,610        653      +13.4%  
  Ene 2024     ₡   7,410,372        647       -2.5%    ──────
  Feb 2024     ₡   6,527,940        591      -11.9%  
  Mar 2024     ₡   8,550,041        734      +31.0%  
  Abr 2024     ₡   7,579,428        664      -11.4%  
  May 2024     ₡   8,609,028        737      +13.6%  
  Jun 2024     ₡   8,070,671        649       -6.3%  
  Jul 2024     ₡   9,050,392        750      +12.1%  
  Ago 2024     ₡   7,237,882        787      -20.0%  
  Sep 2024     ₡   7,239,518        742       +0.0%  
  Oct 2024     ₡   7,137,061        677       -1.4%  
  Nov 2024     ₡   7,686,408        697       +7.7%  
  Dic 2024     ₡

##  Estrategia para generar ventas desde el Día 1
### Basada 100% en datos reales del análisis

---

###  Punto de Equilibrio Mensual — La Victoria

| Categoría | Concepto | Monto (₡) |
|---|---|---|
| Operativo | Alquiler + Luz + Agua + Internet + Basura | ₡273,000 |
| Servicios | Sistema + Contabilidad + IA + Teléfono | ₡69,600 |
| Seguros | Rosa + Charlie + Póliza | ₡105,000 |
| Salarios | Rosa + Charlie (₡100,000/semana c/u) | ₡800,000 |
| | **TOTAL GASTOS FIJOS** | **₡1,247,600** |

#### Cálculo del mínimo para sobrevivir:Margen promedio estimado: 35%
₡1,247,600 ÷ 35% = ₡3,564,571 en ventas mínimas/mes
= 322 facturas al mes
= 11 facturas/día (abriendo 30 días)
= 14 facturas/día (abriendo 23 días hábiles)

>  El negocio anterior hacía **30+ facturas diarias** en sus mejores meses.  
> El punto de equilibrio es **totalmente alcanzable**.

---

###  Estrategia 1 — Inventario inicial inteligente

De los **10,120 artículos** del catálogo, no se necesitan todos.  
Se necesitan los **correctos**. Prioridad día 1:

| Prioridad | Categoría | Razón |
|---|---|---|
|  1 | Filtros (aceite, aire, combustible) | Mayor volumen de artículos y alta rotación |
|  2 | Pastillas y zapatas de freno | Demanda constante, urgente |
|  3 | Eléctrico (relays, fusibles, carbones) | Tickets más altos |
| 4 | Hules y bushings | Alta frecuencia de compra |
| 5 | Roles y rodamientos | Críticos, el cliente no puede esperar |

---

###  Estrategia 2 — El momento ideal para abrir

Los datos confirman que **Enero y Febrero son los mejores meses**:

| Mes | Ventas | Observación |
|---|---|---|
| Febrero 2025 | ₡9,889,646 |  Récord absoluto histórico |
| Enero 2025 | ₡9,828,130 |  Segundo mejor mes |
| Julio 2024 | ₡9,050,392 | Tercer mejor mes |

>  **¿Por qué enero/febrero?**  
> La gente hace mantenimiento preventivo después de las fiestas.  
> Abrir en Q1 significa **arrancar con viento a favor**.
>Julio estan las pre tiempo de fechas de vacaciones de medio año, cuando las personas sale de viaje
---

###  Estrategia 3 — Capturar clientes desde el inicio

El negocio anterior tenía **3,077 clientes únicos en 2024**.  
Esos clientes ya compraban repuestos en la zona.

**Plan de captura:**
- WhatsApp activo desde el **día 1** con catálogo y precios
- Precio competitivo en **filtros y aceites** (productos gancho)
- Disponibilidad inmediata en **frenos y embragues** (no pueden esperar)
- Base de datos de clientes desde la **primera venta**

---

###  Metas realistas — Mes 1

| Meta | Facturas/día | Ventas mensuales |
|---|---|---|
| 🔴 Mínima (sobrevivir) | 14 | ₡3,600,000 |
| 🟡 Cómoda | 20 | ₡5,000,000 |
| 🟢 Ambiciosa | 28 | ₡7,000,000 |

>  **Referencia clave:** El negocio de referencia  
> **nunca bajó de ₡6.5M mensual** en ningún mes completo de 2024.  
> La Victoria tiene una base sólida para planificar sobre eso.

---

###  Lecciones aprendidas del negocio de referencia

| Problema detectado | Solución para La Victoria |
|---|---|
| 45% de ventas dependían de Charlie M. | Rosa debe ser la vendedora principal del mostrador |
| 4% de facturas sin vendedor asignado | Registrar **siempre** quién realizó cada venta |
| Sin detalle de productos por factura | El sistema debe registrar cada artículo vendido |
| 880 facturas sin vendedor (nan) | Campo vendedor **obligatorio** en el sistema |

---
*Fuente: Análisis de datos reales Sep 2023 → Sep 2025*  
*Gastos fijos: Archivo Gastos_Fijos_La_Victoria.xlsx*

In [25]:
# ═════════════════════════ — ANÁLISIS DEL CATÁLOGO DE ARTÍCULOS
# Este es el análisis más valioso para definir
# qué debe tener La Victoria en inventario.
# ══════════════════════════════════════════════════

print("ANÁLISIS DEL CATÁLOGO DE ARTÍCULOS")
print("=" * 60)

# Estado general del catálogo
total_articulos  = len(df_articulos)
activos          = len(df_articulos[df_articulos['Estado'] == 'Activo'])
con_stock        = len(df_articulos[df_articulos['Inventario'] > 0])
sin_stock        = len(df_articulos[df_articulos['Inventario'] <= 0])
valor_inventario = (df_articulos['Inventario'] * df_articulos['Costo neto']).sum()

print(f"\nRESUMEN GENERAL DEL CATÁLOGO:")
print(f"  Total artículos registrados:    {total_articulos:>8,}")
print(f"  Artículos activos:              {activos:>8,}")
print(f"  Artículos con stock físico:     {con_stock:>8,}")
print(f"  Artículos sin stock:            {sin_stock:>8,}")
print(f"  Valor total en inventario:      {valor_inventario:>12,.0f}")

# Porcentaje de artículos sin stock
pct_sin_stock = sin_stock / total_articulos * 100
print(f"\n  Porcentaje sin stock:           {pct_sin_stock:>7.1f}%")
print(f"  Porcentaje con stock:           {100-pct_sin_stock:>7.1f}%")

ANÁLISIS DEL CATÁLOGO DE ARTÍCULOS

RESUMEN GENERAL DEL CATÁLOGO:
  Total artículos registrados:      10,120
  Artículos activos:                10,094
  Artículos con stock físico:        3,183
  Artículos sin stock:               6,937
  Valor total en inventario:        20,807,162

  Porcentaje sin stock:              68.5%
  Porcentaje con stock:              31.5%


In [36]:
# ════════════════════════════════ — ANÁLISIS POR CATEGORIA
# Identificar cuáles categorías tienen más
# artículos, mejor margen y mayor valor
# ══════════════════════════════════════════════════

print("ANÁLISIS POR CATEGORIA DE PRODUCTO")
print("=" * 75)

# Agrupar por Línea (categoría del artículo)
por_categoria = df_articulos.groupby('Línea').agg(
    total_articulos  = ('Código', 'count'),
    con_stock        = ('Inventario', lambda x: (x > 0).sum()),
    unidades_stock   = ('Inventario', 'sum'),
    precio_promedio  = ('Precio venta', 'mean'),
    costo_promedio   = ('Costo neto', 'mean'),
    margen_promedio  = ('%Utilidad', 'mean'),
).reset_index()

# Calcular valor del inventario por categoría
por_categoria['valor_inventario'] = (
    por_categoria['unidades_stock'] * por_categoria['costo_promedio']
)

# Calcular porcentaje de artículos con stock
por_categoria['pct_con_stock'] = (
    por_categoria['con_stock'] / por_categoria['total_articulos'] * 100
).round(1)

# Ordenar por total de artículos
por_categoria = por_categoria[
    por_categoria['total_articulos'] >= 5
].sort_values('total_articulos', ascending=False).reset_index(drop=True)

print(f"\n{'#':<3} {'Categoria':<35} {'Articulos':>9} {'Con Stock':>9} {'Margen%':>8} {'Valor Inv':>14}")
print("-" * 82)

for i, row in por_categoria.iterrows():
    categoria = str(row['Línea'])[:33]
    print(f"{i+1:<3} {categoria:<35} "
          f"{int(row['total_articulos']):>9,} "
          f"{int(row['con_stock']):>9,} "
          f"{row['margen_promedio']:>7.1f}% "
          f"  {row['valor_inventario']:>12,.0f}")

ANÁLISIS POR CATEGORIA DE PRODUCTO

#   Categoria                           Articulos Con Stock  Margen%      Valor Inv
----------------------------------------------------------------------------------
1   PARTES DE CARROSERIA                      895       132    82.7%     24,010,147
2   BEARING                                   620        85    90.1%     29,045,638
3   DEPARTAMENTO DE ELECTRO                   514       164    83.4%     19,640,523
4   TUERCAS Y TORNILLOS                       367       289   112.6%      9,378,262
5   ACCESORIOS                                332       114    70.5%     10,836,084
6   FAJAS DE ABANICO                          312       109    81.2%        827,020
7   PASTILLAS DE FRENO                        276        84    84.4%        869,594
8   BUSCHIN DE TIJERETA /RESORTE,TENS         269       182    86.8%      1,803,361
9   ADICTIVOS GENERAL                         259       101    50.7%        722,455
10  MOTOR                                

In [30]:
# ══════════════════════— RENTABILIDAD POR CATEGORIA
# Identificar cuáles categorías dejan más ganancia
# ══════════════════════════════════════════════════

print("RENTABILIDAD POR CATEGORIA — ORDENADO POR MARGEN")
print("=" * 65)

# Filtrar categorías con margen válido y suficientes artículos
rentabilidad = por_categoria[
    (por_categoria['margen_promedio'] > 0) &
    (por_categoria['total_articulos'] >= 5)
].sort_values('margen_promedio', ascending=False).reset_index(drop=True)

print(f"\n{'#':<3} {'Categoria':<35} {'Margen%':>8} {'Precio Prom':>12} {'Costo Prom':>12}")
print("-" * 75)

for i, row in rentabilidad.iterrows():
    categoria = str(row['Línea'])[:33]
    # Clasificar el margen
    if row['margen_promedio'] >= 60:
        nivel = "ALTO"
    elif row['margen_promedio'] >= 35:
        nivel = "MEDIO"
    else:
        nivel = "BAJO"

    print(f"{i+1:<3} {categoria:<35} "
          f"{row['margen_promedio']:>7.1f}%  "
          f"  {row['precio_promedio']:>10,.0f}  "
          f"  {row['costo_promedio']:>10,.0f}  "
          f"  [{nivel}]")

# Resumen de márgenes
print(f"\nRESUMEN DE MARGENES:")
print(f"  Categorias con margen ALTO  (>=60%): "
      f"{len(rentabilidad[rentabilidad['margen_promedio'] >= 60]):>3}")
print(f"  Categorias con margen MEDIO (35-60%): "
      f"{len(rentabilidad[(rentabilidad['margen_promedio'] >= 35) & (rentabilidad['margen_promedio'] < 60)]):>3}")
print(f"  Categorias con margen BAJO  (<35%):  "
      f"{len(rentabilidad[rentabilidad['margen_promedio'] < 35]):>3}")

RENTABILIDAD POR CATEGORIA — ORDENADO POR MARGEN

#   Categoria                            Margen%  Precio Prom   Costo Prom
---------------------------------------------------------------------------
1   EXPRES Y MAS                        31448.7%        41,584           524    [ALTO]
2   MANILLAS                            11351.0%         9,747         5,322    [ALTO]
3   HULES DE ESTABILISADORA               140.6%         2,420         1,078    [ALTO]
4   RETENEDORES ESPECIALES                136.6%         8,726         4,072    [ALTO]
5   HULES MUFLA,Y HULES MAS               135.8%         2,469         1,241    [ALTO]
6   FARROL DIRECIONAL                     123.5%        14,167         6,475    [ALTO]
7   RETENEDORES NJ000-100                 123.2%         4,206         1,843    [ALTO]
8   HULES DE CAJAS DE CREMALLERA          117.8%         3,298         1,433    [ALTO]
9   EMPAQUE TAPA VALVULA                  116.6%         9,022         4,131    [ALTO]
10  ROTULAS SUSP

In [29]:
# ══════════════════ — ARTICULOS PRIORITARIOS PARA LA VICTORIA
# Combina: tiene stock + buen margen + precio accesible
# Estos son los articulos que DEBEN estar el dia 1
# ══════════════════════════════════════════════════

print("ARTICULOS PRIORITARIOS PARA INVENTARIO INICIAL")
print("LA VICTORIA — DIA 1")
print("=" * 70)

# Categorías estratégicas definidas por el análisis
categorias_prioritarias = [
    'FILTROS',
    'PASTILLAS DE FRENO',
    'ELECTRICO',
    'HULES Y BUSHINGS',
    'ROLES',
    'FRENO',
    'CONJUNTO DE CLUTCH',
    'ADICTIVOS',
    'BOMBAS',
    'FAJAS',
    'ROTULAS',
    'RETENEDORES',
    'SENSORES',
]

for cat in categorias_prioritarias:
    # Filtrar artículos de esa categoría con stock y precio válido
    df_cat = df_articulos[
        (df_articulos['Línea'].str.upper().str.contains(cat, na=False)) &
        (df_articulos['Precio venta'] > 0) &
        (df_articulos['%Utilidad'] > 0)
    ].copy()

    if len(df_cat) == 0:
        continue

    total_cat    = len(df_cat)
    con_stock_cat = len(df_cat[df_cat['Inventario'] > 0])
    margen_cat   = df_cat['%Utilidad'].mean()
    precio_cat   = df_cat['Precio venta'].mean()

    print(f"\n  {cat}")
    print(f"  {'-'*50}")
    print(f"  Articulos en catalogo:  {total_cat:>6,}")
    print(f"  Con stock actual:       {con_stock_cat:>6,}")
    print(f"  Margen promedio:        {margen_cat:>6.1f}%")
    print(f"  Precio promedio:        {precio_cat:>10,.0f}")

    # Top 5 articulos de esa categoría por precio de venta
    top5 = df_cat.nlargest(5, 'Precio venta')[
        ['Nombre', 'Precio venta', 'Costo neto', '%Utilidad', 'Inventario']
    ]
    print(f"\n  Top 5 por precio de venta:")
    for _, art in top5.iterrows():
        nombre = str(art['Nombre'])[:55]
        print(f"    - {nombre}")
        print(f"      Precio: {art['Precio venta']:>10,.0f}  "
              f"Costo: {art['Costo neto']:>10,.0f}  "
              f"Margen: {art['%Utilidad']:>6.1f}%  "
              f"Stock: {float(art['Inventario']):>5.0f}")

ARTICULOS PRIORITARIOS PARA INVENTARIO INICIAL
LA VICTORIA — DIA 1

  FILTROS
  --------------------------------------------------
  Articulos en catalogo:     338
  Con stock actual:          188
  Margen promedio:          75.8%
  Precio promedio:            10,478

  Top 5 por precio de venta:
    - RECONSTRUIDASUZUKI SIDECKICK 1,8L 96-98 VITARA 99-08 1,
      Precio:     76,840  Costo:     40,000  Margen:   70.0%  Stock:     0
    - FILTRO DE CAJA AUTOMATICA  TY HILUX, FORTUNE, PRADO
      Precio:     59,435  Costo:     17,000  Margen:  209.4%  Stock:     1
    - Nissan Pickup 2WD (1990-97), Frontier (2006-07)  (Nissa
      Precio:     45,000  Costo:     30,820  Margen:   29.2%  Stock:     0
    - FILTRO SERVICIO PESADO ACEITE HIDRAHULICO new holland 4
      Precio:     45,000  Costo:     23,795  Margen:   67.4%  Stock:     0
    - PATILLA CLUTCH TY RAV4 00-
      Precio:     44,781  Costo:     23,311  Margen:   70.0%  Stock:     0

  PASTILLAS DE FRENO
  --------------------------

In [31]:
# ═════════════ — LOS 50 ARTICULOS INDISPENSABLES
# Criterios de selección:
# 1. Categorías de alta rotación del negocio de referencia
# 2. Margen entre 20% mínimo y sin techo máximo
# 3. Aplicación a las marcas predominantes del mercado
# 4. Cubre los 3 tipos de cliente (dueño, mecánico, taller)
# ══════════════════════════════════════════════════

import pandas as pd

# Filtrar artículos con condiciones mínimas de calidad
df_validos = df_articulos[
    (df_articulos['Precio venta'] > 0) &
    (df_articulos['Costo neto'] > 0) &
    (df_articulos['%Utilidad'] >= 20) &
    (df_articulos['Estado'] == 'Activo')
].copy()

# Definir categorías prioritarias con su peso estratégico
categorias_peso = {
    'FILTROS':               10,
    'PASTILLAS DE FRENO':     8,
    'FRENO':                  7,
    'CONJUNTO DE CLUTCH':     7,
    'ELECTRICO':              6,
    'HULES Y BUSHINGS':       6,
    'ROLES':                  5,
    'ADICTIVOS':              5,
    'BOMBAS':                 5,
    'FAJAS':                  4,
    'ROTULAS':                4,
    'RETENEDORES':            4,
    'SENSORES':               3,
    'EMPAQUES':               3,
}

seleccionados = []

for categoria, cupo in categorias_peso.items():
    df_cat = df_validos[
        df_validos['Línea'].str.contains(categoria, na=False)
    ].copy()

    if len(df_cat) == 0:
        continue

    # Dentro de cada categoría, priorizar por margen alto
    # y precio de venta razonable (accesible para el cliente)
    df_cat = df_cat.sort_values(
        ['%Utilidad', 'Precio venta'], 
        ascending=[False, True]
    ).head(cupo)

    df_cat['Categoria'] = categoria
    seleccionados.append(df_cat)

df_top50 = pd.concat(seleccionados, ignore_index=True)
df_top50 = df_top50.head(50)

# Agregar número de orden
df_top50.insert(0, 'N', range(1, len(df_top50) + 1))

print("LOS 50 ARTICULOS INDISPENSABLES — TOTAL AUTOMOTRIZ LA VICTORIA")
print("=" * 90)
print(f"\n{'N':<4} {'Categoria':<25} {'Articulo':<45} {'Precio':>10} {'Margen':>8}")
print("-" * 95)

for _, row in df_top50.iterrows():
    nombre = str(row['Nombre'])[:43]
    cat    = str(row['Categoria'])[:23]
    print(f"{int(row['N']):<4} {cat:<25} {nombre:<45} "
          f"  {row['Precio venta']:>8,.0f}  "
          f"{row['%Utilidad']:>7.1f}%")

print(f"\nTOTAL ARTICULOS SELECCIONADOS: {len(df_top50)}")
print(f"Margen promedio de la seleccion: {df_top50['%Utilidad'].mean():.1f}%")
print(f"Precio promedio de la seleccion: {df_top50['Precio venta'].mean():,.0f}")
print(f"Inversion estimada en inventario inicial:")
inversion = (df_top50['Costo neto'] * 1).sum()
print(f"  (1 unidad por articulo): {inversion:>12,.0f}")
print(f"  (2 unidades por articulo): {inversion*2:>12,.0f}")
print(f"  (3 unidades por articulo): {inversion*3:>12,.0f}")

LOS 50 ARTICULOS INDISPENSABLES — TOTAL AUTOMOTRIZ LA VICTORIA

N    Categoria                 Articulo                                          Precio   Margen
-----------------------------------------------------------------------------------------------
1    FILTROS                   BASE DE BOMBILLOS TODO VIDRIO C/CABLE              2,000   1669.9%
2    FILTROS                   FILTRO ACEITE PH2808/PH3593 ALTO KILOMETRAJ        8,500    599.1%
3    FILTROS                   Empaque Cabezot - FRONTIER VG33E D22 (98/05       26,000    360.2%
4    FILTROS                   FILTRO AIRE ACONDICIONADO CABINA NISSAN TIL        9,000    296.4%
5    FILTROS                   ROTULA  ROT.DIR. LH-RH EXT / HONDA-CR-V 4X2       13,000    265.2%
6    FILTROS                   FILTRO DE COMBUSTIBLE LFF5950                      7,800    245.1%
7    FILTROS                   FILTRO DE COMBUSTIBLE LP-5597                      5,800    242.2%
8    FILTROS                   FILTRO DIECEL BF-215/9-132

In [32]:
# ═════════════════ ARTICULOS MAS VENDIDOS
# Metodologia: Rotacion de stock
# Logica: Si un articulo tiene stock en 0 o muy bajo,
# es porque se vendio. Los que mas se agotaron
# son los que mas demanda tuvieron en el mercado.
# ══════════════════════════════════════════════════

print("ARTICULOS MAS VENDIDOS POR ROTACION DE STOCK")
print("Negocio de referencia — Rio Frio")
print("=" * 75)

# Clasificar cada articulo por nivel de stock
def clasificar_stock(inv):
    if inv == 0:
        return 'AGOTADO — Alta rotacion'
    elif inv <= 2:
        return 'CRITICO — Rotacion alta'
    elif inv <= 5:
        return 'BAJO — Rotacion media'
    else:
        return 'DISPONIBLE — Rotacion normal'

df_articulos['nivel_stock'] = df_articulos['Inventario'].apply(clasificar_stock)

# Contar por nivel
resumen_stock = df_articulos.groupby('nivel_stock').agg(
    cantidad = ('Código', 'count'),
).reset_index()
resumen_stock['porcentaje'] = (
    resumen_stock['cantidad'] / len(df_articulos) * 100
).round(1)

print(f"\nDISTRIBUCION GENERAL DE STOCK:")
print(f"  {'Nivel':<35} {'Articulos':>10} {'Porcentaje':>12}")
print(f"  {'-'*60}")
for _, row in resumen_stock.iterrows():
    print(f"  {row['nivel_stock']:<35} {int(row['cantidad']):>10,} {row['porcentaje']:>11.1f}%")

# Articulos agotados con buen margen — estos son los que mas rotaron
df_agotados = df_articulos[
    (df_articulos['Inventario'] == 0) &
    (df_articulos['%Utilidad'] >= 20) &
    (df_articulos['Precio venta'] > 0) &
    (df_articulos['Estado'] == 'Activo')
].copy()

print(f"\n  Total articulos agotados con margen >= 20%: {len(df_agotados):,}")
print(f"  Estos son los candidatos de mayor rotacion")

ARTICULOS MAS VENDIDOS POR ROTACION DE STOCK
Negocio de referencia — Rio Frio

DISTRIBUCION GENERAL DE STOCK:
  Nivel                                Articulos   Porcentaje
  ------------------------------------------------------------
  AGOTADO — Alta rotacion                  6,560        64.8%
  BAJO — Rotacion media                      483         4.8%
  CRITICO — Rotacion alta                  2,567        25.4%
  DISPONIBLE — Rotacion normal               506         5.0%

  Total articulos agotados con margen >= 20%: 6,211
  Estos son los candidatos de mayor rotacion


In [33]:
# ═════════ TOP ARTICULOS AGOTADOS POR CATEGORIA
# Agotado + buen margen = alta rotacion real
# Estos son exactamente los que La Victoria
# debe tener siempre en stock
# ══════════════════════════════════════════════════

print("TOP ARTICULOS AGOTADOS POR CATEGORIA")
print("Estos son los que mas rotaron en el negocio de referencia")
print("=" * 80)

categorias_analizar = [
    'FILTROS',
    'PASTILLAS DE FRENO',
    'FRENO',
    'CONJUNTO DE CLUTCH',
    'ELECTRICO',
    'HULES Y BUSHINGS',
    'ROLES',
    'ADICTIVOS',
    'BOMBAS',
    'FAJAS',
    'ROTULAS',
    'RETENEDORES',
    'EMPAQUES',
    'SENSORES',
]

resultado_categorias = []

for categoria in categorias_analizar:
    df_cat = df_agotados[
        df_agotados['Línea'].str.contains(categoria, na=False)
    ].copy()

    if len(df_cat) == 0:
        continue

    total_agotados = len(df_cat)
    margen_prom    = df_cat['%Utilidad'].mean()
    precio_prom    = df_cat['Precio venta'].mean()

    resultado_categorias.append({
        'Categoria':       categoria,
        'Agotados':        total_agotados,
        'Margen_prom':     margen_prom,
        'Precio_prom':     precio_prom,
    })

df_resultado = pd.DataFrame(resultado_categorias)
df_resultado = df_resultado.sort_values('Agotados', ascending=False).reset_index(drop=True)

print(f"\n{'#':<3} {'Categoria':<28} {'Agotados':>9} {'Margen Prom':>12} {'Precio Prom':>12}")
print(f"  {'-'*68}")

for i, row in df_resultado.iterrows():
    print(f"  {i+1:<3} {row['Categoria']:<28} "
          f"{int(row['Agotados']):>9,} "
          f"{row['Margen_prom']:>11.1f}% "
          f"  {row['Precio_prom']:>10,.0f}")

TOP ARTICULOS AGOTADOS POR CATEGORIA
Estos son los que mas rotaron en el negocio de referencia

#   Categoria                     Agotados  Margen Prom  Precio Prom
  --------------------------------------------------------------------
  1   FRENO                              594        86.7%       18,311
  2   ROTULAS                            300        82.9%       18,197
  3   FAJAS                              268        81.9%       12,204
  4   RETENEDORES                        198        98.9%        7,964
  5   PASTILLAS DE FRENO                 184        89.5%       19,128
  6   SENSORES                           155        69.7%       39,317
  7   ELECTRICO                          155        69.7%       39,317
  8   FILTROS                            148        83.1%       13,771
  9   ADICTIVOS                          131        52.5%        5,693
  10  EMPAQUES                           116       100.0%       14,084
  11  BOMBAS                              69        65

In [34]:
# ═══════════════ DETALLE DE ARTICULOS POR CATEGORIA
# Top 10 articulos agotados de cada categoria
# ordenados por margen de utilidad
# ══════════════════════════════════════════════════

print("DETALLE DE ARTICULOS AGOTADOS POR CATEGORIA")
print("Top 10 por categoria — ordenados por margen")
print("=" * 80)

for categoria in categorias_analizar:
    df_cat = df_agotados[
        df_agotados['Línea'].str.contains(categoria, na=False)
    ].sort_values('%Utilidad', ascending=False).head(10)

    if len(df_cat) == 0:
        continue

    print(f"\n  {categoria}")
    print(f"  {'-'*75}")
    print(f"  {'Articulo':<50} {'Precio':>10} {'Margen':>8}")
    print(f"  {'-'*75}")

    for _, row in df_cat.iterrows():
        nombre = str(row['Nombre'])[:48]
        print(f"  {nombre:<50} "
              f"  {row['Precio venta']:>8,.0f} "
              f"  {row['%Utilidad']:>6.1f}%")

DETALLE DE ARTICULOS AGOTADOS POR CATEGORIA
Top 10 por categoria — ordenados por margen

  FILTROS
  ---------------------------------------------------------------------------
  Articulo                                               Precio   Margen
  ---------------------------------------------------------------------------
  BASE DE BOMBILLOS TODO VIDRIO C/CABLE                   2,000   1669.9%
  FILTRO ACEITE PH2808/PH3593 ALTO KILOMETRAJE            8,500    599.1%
  ROTULA  ROT.DIR. LH-RH EXT / HONDA-CR-V 4X2 07-1       13,000    265.2%
  FILTRO DE COMBUSTIBLE LP-5597                           5,800    242.2%
  FILTRO DIECEL BF-215/9-13240-032-0 NISSAN JUNIOR        3,800    236.3%
  ROTULA SUS. LH INF / HONDA-CR-V 4X2 07-11              13,000    221.2%
  FILTRO AIRE HY CRETA                                   14,500    185.2%
  FILTRO CAJA AUTOMATICA MB MONTERO SPORT                18,000    183.9%
  BOCINA DEL. NISSAN B12                                 23,500    181.0%
  FILT

In [35]:
# ═══════════════════════ RESUMEN EJECUTIVO
# Conclusion del analisis de rotacion
# ══════════════════════════════════════════════════

print("RESUMEN EJECUTIVO — ANALISIS DE ROTACION")
print("=" * 65)

total_activos  = len(df_articulos[df_articulos['Estado'] == 'Activo'])
total_agotados = len(df_agotados)
pct_agotados   = total_agotados / total_activos * 100

print(f"\n  Total articulos activos:          {total_activos:>8,}")
print(f"  Articulos agotados con margen:    {total_agotados:>8,}")
print(f"  Porcentaje agotado:               {pct_agotados:>8.1f}%")

print(f"\n  CATEGORIA MAS AGOTADA (mayor rotacion):")
top_cat = df_resultado.iloc[0]
print(f"  {top_cat['Categoria']}")
print(f"  {int(top_cat['Agotados'])} articulos agotados")
print(f"  Margen promedio: {top_cat['Margen_prom']:.1f}%")

print(f"\n  CONCLUSION PARA LA VICTORIA:")
print(f"  Las categorias con mayor rotacion deben tener")
print(f"  stock permanente minimo de 2 a 3 unidades por articulo.")
print(f"  Los articulos agotados en el negocio de referencia")
print(f"  son exactamente los que La Victoria debe priorizar")
print(f"  en su inventario inicial.")

print(f"\n  INVERSION ESTIMADA EN STOCK MINIMO:")
inversion_min = (df_agotados['Costo neto'].sum()) * 1
inversion_rec = (df_agotados['Costo neto'].sum()) * 2
print(f"  1 unidad por articulo agotado:   {inversion_min:>12,.0f}")
print(f"  2 unidades por articulo agotado: {inversion_rec:>12,.0f}")

RESUMEN EJECUTIVO — ANALISIS DE ROTACION

  Total articulos activos:            10,094
  Articulos agotados con margen:       6,211
  Porcentaje agotado:                   61.5%

  CATEGORIA MAS AGOTADA (mayor rotacion):
  FRENO
  594 articulos agotados
  Margen promedio: 86.7%

  CONCLUSION PARA LA VICTORIA:
  Las categorias con mayor rotacion deben tener
  stock permanente minimo de 2 a 3 unidades por articulo.
  Los articulos agotados en el negocio de referencia
  son exactamente los que La Victoria debe priorizar
  en su inventario inicial.

  INVERSION ESTIMADA EN STOCK MINIMO:
  1 unidad por articulo agotado:     82,663,204
  2 unidades por articulo agotado:  165,326,409


In [38]:
# ══════════════════════════════════════════════════
# CELDA 22 — ARTICULOS CRITICOS DE EMERGENCIA
# Definicion: El cliente los necesita hoy.
# Si no los tenes, se va a la competencia.
# Orden de criticidad definido por experiencia
# del propietario de La Victoria.
# ══════════════════════════════════════════════════

print("ANALISIS 3 — ARTICULOS CRITICOS DE EMERGENCIA")
print("Total Automotriz La Victoria")
print("=" * 75)

# ── DEFINICION DE GRUPOS CRITICOS ──────────────────
# Orden: 1=mas critico, 4=menos critico
grupos_criticos = {

    1: {
        'nombre'     : 'GRUPO 1 — EL VEHICULO NO ARRANCA',
        'descripcion': 'El cliente no puede moverse. Necesita solucion inmediata.',
        'keywords'   : [
            'BOMBA COMBUSTIBLE', 'BOMBA DE COMBUSTIBLE',
            'ALTERNADOR', 'AUTOMATICO', 'ARRANC',
            'RELAY', 'RELE', 'FUSIBLE',
            'BUJIA', 'BOBINA', 'DISTRIBUIDOR',
            'CABLE BUJIA', 'PLATINO', 'CONDENSADOR',
            'CARBON ARRANC', 'CARBON ALTERN',
            'REGULADOR', 'PORTA CARBON',
        ]
    },

    2: {
        'nombre'     : 'GRUPO 2 — EL VEHICULO NO FRENA',
        'descripcion': 'Riesgo de seguridad inmediato. No puede circular.',
        'keywords'   : [
            'PASTILLA', 'ZAPATA', 'TAMBOR',
            'DISCO DE FRENO', 'CILINDRO DE FRENO',
            'CILINDRO RUEDA', 'CILINDRO MAESTRO',
            'LIQUIDO DE FRENO', 'LIQUIDO FRENOS',
            'MANGUERA DE FRENO', 'MANGUERA FRENO',
            'EMPAQUE FRENO', 'KIT FRENO',
            'SERVO FRENO', 'BOSTER',
        ]
    },

    3: {
        'nombre'     : 'GRUPO 3 — EL VEHICULO NO TRANSMITE POTENCIA',
        'descripcion': 'El vehiculo avanza mal o no avanza. No es util para el cliente.',
        'keywords'   : [
            'BOTA DE EJE', 'BOTA EJE', 'BOTA INT',
            'CRUCETA', 'RODAMIENTO RUEDA', 'ROLL',
            'KIT CLUTCH', 'CONJUNTO CLUTCH', 'PLATO CLUTCH',
            'DISCO CLUTCH', 'COLLARÍN', 'COLARIN',
            'CAJA VELOCIDAD', 'SINCRONIZADO',
            'DIFERENCIAL', 'CATALINA',
        ]
    },

    4: {
        'nombre'     : 'GRUPO 4 — EL VEHICULO PIERDE FLUIDOS',
        'descripcion': 'Si no se atiende a tiempo puede generar daño mayor al motor.',
        'keywords'   : [
            'EMPAQUE CULATA', 'JUNTA CULATA',
            'EMPAQUE TAPA VALVULA', 'EMPAQUE CARTER',
            'RETENEDOR', 'SELLO', 'RETÉN',
            'MANGUERA RADIADOR', 'TAPON RADIADOR',
            'TAPON CARTER', 'TAPON ACEITE',
            'BOMBA AGUA', 'BOMBA DE AGUA',
            'RADIADOR', 'TERMOSTATO',
        ]
    },
}

# ── BUSCAR ARTICULOS POR GRUPO ──────────────────────
resumen_grupos = []

for num_grupo, info in grupos_criticos.items():
    print(f"\n{'='*75}")
    print(f"  {info['nombre']}")
    print(f"  {info['descripcion']}")
    print(f"{'='*75}")

    articulos_grupo = []

    for keyword in info['keywords']:
        coincidencias = df_articulos[
            (df_articulos['Nombre'].str.upper().str.contains(keyword, na=False)) &
            (df_articulos['Precio venta'] > 0) &
            (df_articulos['%Utilidad'] >= 20) &
            (df_articulos['Estado'] == 'Activo')
        ].copy()
        coincidencias['keyword'] = keyword
        articulos_grupo.append(coincidencias)

    if not articulos_grupo:
        print(f"  Sin articulos encontrados en catalogo")
        continue

    df_grupo = pd.concat(articulos_grupo).drop_duplicates(subset='Código')
    df_grupo = df_grupo.sort_values('%Utilidad', ascending=False)

    total_grupo    = len(df_grupo)
    agotados_grupo = len(df_grupo[df_grupo['Inventario'] == 0])
    margen_prom    = df_grupo['%Utilidad'].mean()
    inversion_min  = df_grupo['Costo neto'].sum()

    print(f"\n  Articulos encontrados en catalogo:  {total_grupo:>6,}")
    print(f"  Actualmente agotados:               {agotados_grupo:>6,}")
    print(f"  Margen promedio del grupo:          {margen_prom:>6.1f}%")
    print(f"  Inversion minima (1 und c/u):       {inversion_min:>10,.0f}")

    # Top 15 articulos del grupo
    top15 = df_grupo.head(15)
    print(f"\n  TOP 15 ARTICULOS DEL GRUPO:")
    print(f"  {'Articulo':<52} {'Precio':>10} {'Margen':>8} {'Stock':>7}")
    print(f"  {'-'*82}")

    for _, row in top15.iterrows():
        nombre = str(row['Nombre'])[:50]
        stock_txt = 'AGOTADO' if row['Inventario'] == 0 else str(int(row['Inventario']))
        print(f"  {nombre:<52} "
              f"  {row['Precio venta']:>8,.0f} "
              f"  {row['%Utilidad']:>6.1f}% "
              f"  {stock_txt:>7}")

    resumen_grupos.append({
        'Grupo'         : info['nombre'],
        'Total articulos': total_grupo,
        'Agotados'      : agotados_grupo,
        'Margen prom'   : round(margen_prom, 1),
        'Inversion min' : round(inversion_min, 0),
    })

ANALISIS 3 — ARTICULOS CRITICOS DE EMERGENCIA
Total Automotriz La Victoria

  GRUPO 1 — EL VEHICULO NO ARRANCA
  El cliente no puede moverse. Necesita solucion inmediata.

  Articulos encontrados en catalogo:     514
  Actualmente agotados:                  354
  Margen promedio del grupo:            88.1%
  Inversion minima (1 und c/u):        5,955,595

  TOP 15 ARTICULOS DEL GRUPO:
  Articulo                                                 Precio   Margen   Stock
  ----------------------------------------------------------------------------------
  BOBINA NISSAN                                            35,700   3059.3%   AGOTADO
  ALTERNADOR HY TUCSON 05-                                115,000   1490.2%   AGOTADO
  CABLE BUJIAS TY HILUX 3VZE/RUNNER 92- 6CILINDROS         21,800    502.9%   AGOTADO
  CABLE BUJIAS TY RUNNER V6 3,0                            19,000    460.5%   AGOTADO
  CABLE BUJIAS HY ELANTRA NGK DOCH                         30,000    431.0%         1
  RELAY LUCES 

In [39]:
# ═════════════════════ RESUMEN EJECUTIVO GRUPOS CRITICOS
# Vision consolidada de los 4 grupos
# ══════════════════════════════════════════════════

print("RESUMEN EJECUTIVO — ARTICULOS CRITICOS DE EMERGENCIA")
print("=" * 70)

df_resumen = pd.DataFrame(resumen_grupos)

print(f"\n{'Grupo':<45} {'Articulos':>9} {'Agotados':>9} "
      f"{'Margen':>8} {'Inversion min':>14}")
print(f"  {'-'*88}")

total_inversion = 0
for _, row in df_resumen.iterrows():
    grupo_corto = str(row['Grupo'])[:43]
    print(f"  {grupo_corto:<45} "
          f"{int(row['Total articulos']):>9,} "
          f"{int(row['Agotados']):>9,} "
          f"{row['Margen prom']:>7.1f}% "
          f"  {row['Inversion min']:>12,.0f}")
    total_inversion += row['Inversion min']

print(f"\n  {'TOTAL INVERSION MINIMA (1 und x articulo)':<45} "
      f"  {total_inversion:>12,.0f}")
print(f"  {'TOTAL INVERSION RECOMENDADA (2 und x articulo)':<45} "
      f"  {total_inversion*2:>12,.0f}")

print(f"\n\nREGLA DE ORO PARA LA VICTORIA:")
print(f"  Estos articulos NUNCA pueden estar en cero.")
print(f"  Cuando llegue al ultimo en stock, se pide de inmediato.")
print(f"  El costo de perder una venta critica no es solo esa venta —")
print(f"  es perder al cliente para siempre.")

RESUMEN EJECUTIVO — ARTICULOS CRITICOS DE EMERGENCIA

Grupo                                         Articulos  Agotados   Margen  Inversion min
  ----------------------------------------------------------------------------------------
  GRUPO 1 — EL VEHICULO NO ARRANCA                    514       354    88.1%      5,955,595
  GRUPO 2 — EL VEHICULO NO FRENA                      487       348    80.6%      4,533,358
  GRUPO 3 — EL VEHICULO NO TRANSMITE POTENCIA         816       552   297.8%     10,734,687
  GRUPO 4 — EL VEHICULO PIERDE FLUIDOS                775       466   106.3%      6,545,249

  TOTAL INVERSION MINIMA (1 und x articulo)         27,768,889
  TOTAL INVERSION RECOMENDADA (2 und x articulo)     55,537,778


REGLA DE ORO PARA LA VICTORIA:
  Estos articulos NUNCA pueden estar en cero.
  Cuando llegue al ultimo en stock, se pide de inmediato.
  El costo de perder una venta critica no es solo esa venta —
  es perder al cliente para siempre.


 — Analisis de meta salarial

In [43]:
# ══════════════════════════════════════════════════
# CELDA 24 — ESCALERA SALARIAL PROGRESIVA
# Escalones iguales para Charlie y Rosa
# Escalon 1: ₡100,000 semanales c/u
# Escalon 2: ₡125,000 semanales c/u
# Escalon 3: ₡150,000 semanales c/u
# Escalon 4: ₡175,000 semanales c/u
# Escalon 5: ₡200,000 semanales c/u
# ══════════════════════════════════════════════════

print("ESCALERA SALARIAL PROGRESIVA — TOTAL AUTOMOTRIZ LA VICTORIA")
print("Charlie Marenco y Rosa — Mismo salario por escalon")
print("=" * 70)

margen           = 0.35
semanas_mes      = 4
dias_habiles_mes = 23
ticket_promedio  = 11080
promedio_ref     = 7888601  # promedio mensual Rio Frio

# ── GASTOS OPERATIVOS FIJOS ──────────────────────────
gastos_fijos = {
    'Alquiler'               : 226000,
    'Basura'                 : 3000,
    'Luz'                    : 20000,
    'Agua'                   : 6000,
    'Internet'               : 18000,
    'Sistema'                : 22600,
    'Contabilidad'           : 25000,
    'Inteligencia Artificial': 7000,
    'Linea Telefonica'       : 15000,
    'Seguro Rosa'            : 50000,
    'Seguro Charlie'         : 50000,
    'Poliza'                 : 5000,
}

total_operativo = sum(gastos_fijos.values())

print(f"\nGASTOS OPERATIVOS FIJOS (sin salarios): {total_operativo:>12,.0f}")
print(f"Margen de utilidad aplicado:            {margen*100:>11.0f}%")
print(f"Ticket promedio de referencia:          {ticket_promedio:>12,.0f}")
print(f"Promedio mensual negocio Rio Frio:      {promedio_ref:>12,.0f}")

# ── ESCALONES SALARIALES ────────────────────────────
escalones = [
    {'escalon': 1, 'semanal': 100000, 'fase': 'Arranque'},
    {'escalon': 2, 'semanal': 125000, 'fase': 'Estabilizacion'},
    {'escalon': 3, 'semanal': 150000, 'fase': 'Crecimiento'},
    {'escalon': 4, 'semanal': 175000, 'fase': 'Consolidacion'},
    {'escalon': 5, 'semanal': 200000, 'fase': 'Meta final'},
]

print(f"\n\n{'ESC':<5} {'FASE':<18} {'Semanal c/u':>12} {'Salarios/mes':>13} "
      f"{'Total cubrir':>13} {'Ventas min':>13} {'Facts/dia':>10} {'Venta/dia':>12}")
print(f"  {'-'*100}")

tabla_escalones = []

for esc in escalones:
    semanal         = esc['semanal']
    salario_mensual = semanal * semanas_mes        # uno solo
    total_salarios  = salario_mensual * 2          # Charlie + Rosa
    total_necesario = total_operativo + total_salarios
    ventas_min      = total_necesario / margen
    facturas_dia    = (ventas_min / ticket_promedio) / dias_habiles_mes
    venta_dia       = ventas_min / dias_habiles_mes
    pct_vs_ref      = ventas_min / promedio_ref * 100

    tabla_escalones.append({
        'escalon'        : esc['escalon'],
        'fase'           : esc['fase'],
        'semanal'        : semanal,
        'total_salarios' : total_salarios,
        'total_necesario': total_necesario,
        'ventas_min'     : ventas_min,
        'facturas_dia'   : facturas_dia,
        'venta_dia'      : venta_dia,
        'pct_vs_ref'     : pct_vs_ref,
    })

    print(f"  {esc['escalon']:<5} {esc['fase']:<18} "
          f"  {semanal:>10,.0f} "
          f"  {total_salarios:>11,.0f} "
          f"  {total_necesario:>11,.0f} "
          f"  {ventas_min:>11,.0f} "
          f"  {facturas_dia:>9.1f} "
          f"  {venta_dia:>10,.0f}")

# ── DETALLE POR ESCALON ──────────────────────────────
print(f"\n\nDETALLE POR ESCALON:")

for e in tabla_escalones:
    print(f"\n{'='*70}")
    print(f"  ESCALON {e['escalon']} — {e['fase'].upper()}")
    print(f"  Salario: {e['semanal']:,.0f} semanales c/u "
          f"({e['semanal']*semanas_mes:,.0f} mensuales c/u)")
    print(f"{'='*70}")
    print(f"  Salarios totales (Charlie + Rosa):  {e['total_salarios']:>12,.0f}")
    print(f"  Gastos operativos:                  {total_operativo:>12,.0f}")
    print(f"  {'─'*48}")
    print(f"  TOTAL A CUBRIR POR MES:             {e['total_necesario']:>12,.0f}")
    print(f"  VENTAS MINIMAS NECESARIAS:          {e['ventas_min']:>12,.0f}")
    print(f"  {'─'*48}")
    print(f"  Venta minima diaria:                {e['venta_dia']:>12,.0f}")
    print(f"  Facturas diarias necesarias:        {e['facturas_dia']:>12.1f}")
    print(f"  Porcentaje vs Rio Frio:             {e['pct_vs_ref']:>11.1f}%")

    # Margen de seguridad vs Rio Frio
    margen_seguridad = promedio_ref - e['ventas_min']
    print(f"  Margen de seguridad vs Rio Frio:    {margen_seguridad:>12,.0f}")

# ── RESUMEN DE PROGRESION ────────────────────────────
print(f"\n\n{'='*70}")
print(f"  RESUMEN — PROGRESION SALARIAL")
print(f"{'='*70}")
print(f"\n  {'Escalon':<5} {'Fase':<18} {'Ventas min/mes':>15} "
      f"{'Facts/dia':>10} {'% vs Rio Frio':>14}")
print(f"  {'-'*65}")

for e in tabla_escalones:
    print(f"  {e['escalon']:<5} {e['fase']:<18} "
          f"  {e['ventas_min']:>13,.0f} "
          f"  {e['facturas_dia']:>9.1f} "
          f"  {e['pct_vs_ref']:>12.1f}%")

incremento_e1_e5 = tabla_escalones[-1]['ventas_min'] - tabla_escalones[0]['ventas_min']
incremento_facts = tabla_escalones[-1]['facturas_dia'] - tabla_escalones[0]['facturas_dia']

print(f"\n  Diferencia entre Escalon 1 y Escalon 5:")
print(f"  Ventas adicionales necesarias:      {incremento_e1_e5:>12,.0f}")
print(f"  Facturas adicionales por dia:       {incremento_facts:>12.1f}")

print(f"\n  CONCLUSION:")
print(f"  Pasar del Escalon 1 al Escalon 5 requiere")
print(f"  aproximadamente {incremento_facts:.1f} facturas adicionales por dia.")
print(f"  Con el ticket promedio de {ticket_promedio:,.0f} por factura,")
print(f"  la diferencia entre ganar {tabla_escalones[0]['semanal']:,.0f}")
print(f"  y ganar {tabla_escalones[-1]['semanal']:,.0f} semanales es operativamente")
print(f"  muy manejable si el volumen de clientes crece")
print(f"  de forma sostenida desde el primer mes.")

ESCALERA SALARIAL PROGRESIVA — TOTAL AUTOMOTRIZ LA VICTORIA
Charlie Marenco y Rosa — Mismo salario por escalon

GASTOS OPERATIVOS FIJOS (sin salarios):      447,600
Margen de utilidad aplicado:                     35%
Ticket promedio de referencia:                11,080
Promedio mensual negocio Rio Frio:         7,888,601


ESC   FASE                Semanal c/u  Salarios/mes  Total cubrir    Ventas min  Facts/dia    Venta/dia
  ----------------------------------------------------------------------------------------------------
  1     Arranque                100,000       800,000     1,247,600     3,564,571        14.0      154,981
  2     Estabilizacion          125,000     1,000,000     1,447,600     4,136,000        16.2      179,826
  3     Crecimiento             150,000     1,200,000     1,647,600     4,707,429        18.5      204,671
  4     Consolidacion           175,000     1,400,000     1,847,600     5,278,857        20.7      229,516
  5     Meta final              200,000

In [44]:
# ═════════ CLIENTES FRECUENTES
# Identificar quienes compraron mas veces,
# cuanto gastaron y cuando fue su ultima compra.
# Estos son los clientes que La Victoria
# debe contactar desde el primer dia.
# ══════════════════════════════════════════════════

print("ANALISIS DE CLIENTES FRECUENTES")
print("Negocio de referencia — Rio Frio")
print("=" * 70)

# Excluir registros sin nombre de cliente
df_clientes_fact = df_facturas[
    ~df_facturas['cliente'].isin(['NAN', 'NONE', ''])
].copy()

# Agrupar por cliente
resumen_clientes = df_clientes_fact.groupby('cliente').agg(
    total_compras    = ('total', 'count'),
    gasto_total      = ('total', 'sum'),
    ticket_promedio  = ('total', 'mean'),
    primera_compra   = ('fecha', 'min'),
    ultima_compra    = ('fecha', 'max'),
    gasto_maximo     = ('total', 'max'),
).reset_index()

# Calcular dias desde ultima compra
fecha_corte = df_facturas['fecha'].max()
resumen_clientes['dias_sin_comprar'] = (
    fecha_corte - resumen_clientes['ultima_compra']
).dt.days

# Calcular frecuencia promedio en dias entre compras
resumen_clientes['rango_dias'] = (
    resumen_clientes['ultima_compra'] - resumen_clientes['primera_compra']
).dt.days

resumen_clientes['frecuencia_dias'] = (
    resumen_clientes['rango_dias'] / resumen_clientes['total_compras']
).round(0)

# Clasificar clientes por valor
def clasificar_cliente(row):
    if row['total_compras'] >= 20 and row['gasto_total'] >= 100000:
        return 'A — VIP'
    elif row['total_compras'] >= 10 and row['gasto_total'] >= 50000:
        return 'B — Frecuente'
    elif row['total_compras'] >= 5:
        return 'C — Regular'
    else:
        return 'D — Ocasional'

resumen_clientes['clasificacion'] = resumen_clientes.apply(
    clasificar_cliente, axis=1
)

# Ordenar por total de compras
resumen_clientes = resumen_clientes.sort_values(
    'total_compras', ascending=False
).reset_index(drop=True)

# Resumen por clasificacion
print(f"\nDISTRIBUCION DE CLIENTES POR CLASIFICACION:")
print(f"  {'Clasificacion':<20} {'Clientes':>9} {'% del total':>12}")
print(f"  {'-'*45}")
dist = resumen_clientes.groupby('clasificacion')['cliente'].count()
for clase, cantidad in dist.items():
    pct = cantidad / len(resumen_clientes) * 100
    print(f"  {clase:<20} {cantidad:>9,} {pct:>11.1f}%")

print(f"\n  Total clientes unicos: {len(resumen_clientes):,}")

# Top 30 clientes por frecuencia
print(f"\n\nTOP 30 CLIENTES MAS FRECUENTES:")
print(f"  {'#':<4} {'Cliente':<30} {'Compras':>8} {'Gasto Total':>13} "
      f"{'Ticket Prom':>12} {'Dias sin comprar':>17} {'Clase':>15}")
print(f"  {'-'*102}")

for i, row in resumen_clientes.head(30).iterrows():
    nombre = str(row['cliente'])[:28]
    print(f"  {i+1:<4} {nombre:<30} "
          f"{int(row['total_compras']):>8,} "
          f"  {row['gasto_total']:>11,.0f} "
          f"  {row['ticket_promedio']:>10,.0f} "
          f"  {int(row['dias_sin_comprar']):>15,} "
          f"  {row['clasificacion']:>15}")

# Clientes VIP detalle
clientes_vip = resumen_clientes[
    resumen_clientes['clasificacion'] == 'A — VIP'
]
print(f"\n\nCLIENTES VIP — DETALLE COMPLETO:")
print(f"  Estos son los clientes que La Victoria")
print(f"  debe priorizar desde el primer dia de apertura.")
print(f"  {'─'*65}")

for i, row in clientes_vip.iterrows():
    nombre = str(row['cliente'])[:35]
    print(f"\n  {nombre}")
    print(f"  Compras realizadas:      {int(row['total_compras']):>8,}")
    print(f"  Gasto total historico:   {row['gasto_total']:>10,.0f}")
    print(f"  Ticket promedio:         {row['ticket_promedio']:>10,.0f}")
    print(f"  Gasto maximo en una vez: {row['gasto_maximo']:>10,.0f}")
    print(f"  Primera compra:          "
          f"{row['primera_compra'].strftime('%d/%m/%Y'):>12}")
    print(f"  Ultima compra:           "
          f"{row['ultima_compra'].strftime('%d/%m/%Y'):>12}")
    print(f"  Dias sin comprar:        {int(row['dias_sin_comprar']):>8,}")
    print(f"  Frecuencia promedio:     cada {int(row['frecuencia_dias'])} dias")

ANALISIS DE CLIENTES FRECUENTES
Negocio de referencia — Rio Frio

DISTRIBUCION DE CLIENTES POR CLASIFICACION:
  Clasificacion         Clientes  % del total
  ---------------------------------------------
  A — VIP                    151         3.0%
  B — Frecuente              160         3.2%
  C — Regular                322         6.4%
  D — Ocasional            4,399        87.4%

  Total clientes unicos: 5,032


TOP 30 CLIENTES MAS FRECUENTES:
  #    Cliente                         Compras   Gasto Total  Ticket Prom  Dias sin comprar           Clase
  ------------------------------------------------------------------------------------------------------
  1    CHINDO                              308     1,810,068        5,877                 3           A — VIP
  2    JOSE                                198     1,675,079        8,460                 0           A — VIP
  3    MULTISERVICIOS COGO DE SARAP        189     2,382,951       12,608                23           A — VIP
  4

In [46]:
# ═══════════════════════════════DIAS Y MESES DE MAYOR VENTA
# Identificar patrones de cuando vende mas
# el negocio. Esto define:
# - Cuando tener mas stock disponible
# - Cuando programar pedidos a proveedores
# - Cuando necesitas mas personal disponible
# ══════════════════════════════════════════════════

print("ANALISIS DE DIAS Y MESES DE MAYOR VENTA")
print("=" * 65)

# ── POR DIA DE LA SEMANA ────────────────────────────
dias_orden = ['Monday','Tuesday','Wednesday',
              'Thursday','Friday','Saturday','Sunday']
dias_es    = {
    'Monday'   : 'Lunes',
    'Tuesday'  : 'Martes',
    'Wednesday': 'Miercoles',
    'Thursday' : 'Jueves',
    'Friday'   : 'Viernes',
    'Saturday' : 'Sabado',
    'Sunday'   : 'Domingo',
}

por_dia = df_facturas.groupby('dia_semana').agg(
    ventas_total    = ('total', 'sum'),
    facturas        = ('total', 'count'),
    ticket_promedio = ('total', 'mean'),
).reindex([d for d in dias_orden if d in df_facturas['dia_semana'].unique()])

venta_max_dia = por_dia['ventas_total'].max()

print(f"\nVENTAS POR DIA DE LA SEMANA:")
print(f"  {'Dia':<12} {'Ventas Total':>14} {'Facturas':>10} "
      f"{'Ticket Prom':>13} {'Intensidad':>12}")
print(f"  {'-'*65}")

for dia, row in por_dia.iterrows():
    nombre_dia = dias_es.get(dia, dia)
    barra = '|' * int(row['ventas_total'] / venta_max_dia * 20)
    print(f"  {nombre_dia:<12} "
          f"  {row['ventas_total']:>12,.0f} "
          f"  {int(row['facturas']):>8,} "
          f"  {row['ticket_promedio']:>11,.0f} "
          f"  {barra}")

mejor_dia = dias_es.get(por_dia['ventas_total'].idxmax(), '')
peor_dia  = dias_es.get(por_dia['ventas_total'].idxmin(), '')
print(f"\n  Mejor dia de ventas:  {mejor_dia}")
print(f"  Dia mas tranquilo:    {peor_dia}")

# ── POR QUINCENA (inicio vs fin de mes) ─────────────
df_facturas['quincena'] = df_facturas['fecha'].dt.day.apply(
    lambda d: 'Primera quincena (1-15)' if d <= 15 else 'Segunda quincena (16-31)'
)

por_quincena = df_facturas.groupby('quincena').agg(
    ventas_total    = ('total', 'sum'),
    facturas        = ('total', 'count'),
    ticket_promedio = ('total', 'mean'),
).reset_index()

print(f"\n\nVENTAS POR QUINCENA DEL MES:")
print(f"  {'Quincena':<28} {'Ventas Total':>14} {'Facturas':>10} {'Ticket Prom':>13}")
print(f"  {'-'*68}")
for _, row in por_quincena.iterrows():
    print(f"  {row['quincena']:<28} "
          f"  {row['ventas_total']:>12,.0f} "
          f"  {int(row['facturas']):>8,} "
          f"  {row['ticket_promedio']:>11,.0f}")

# ── POR MES DEL AÑO ─────────────────────────────────
meses_es = {1:'Enero',2:'Febrero',3:'Marzo',4:'Abril',
            5:'Mayo',6:'Junio',7:'Julio',8:'Agosto',
            9:'Septiembre',10:'Octubre',11:'Noviembre',12:'Diciembre'}

por_mes = df_facturas.groupby('mes').agg(
    ventas_total    = ('total', 'sum'),
    facturas        = ('total', 'count'),
    ticket_promedio = ('total', 'mean'),
).reset_index()

por_mes['mes_nombre'] = por_mes['mes'].map(meses_es)
venta_max_mes = por_mes['ventas_total'].max()

print(f"\n\nVENTAS POR MES DEL AÑO (historico acumulado):")
print(f"  {'Mes':<12} {'Ventas Total':>14} {'Facturas':>10} "
      f"{'Ticket Prom':>13} {'Intensidad':>12}")
print(f"  {'-'*65}")

por_mes_ord = por_mes.sort_values('mes')
for _, row in por_mes_ord.iterrows():
    barra = '|' * int(row['ventas_total'] / venta_max_mes * 20)
    print(f"  {row['mes_nombre']:<12} "
          f"  {row['ventas_total']:>12,.0f} "
          f"  {int(row['facturas']):>8,} "
          f"  {row['ticket_promedio']:>11,.0f} "
          f"  {barra}")

mejor_mes = por_mes.loc[por_mes['ventas_total'].idxmax(), 'mes_nombre']
peor_mes  = por_mes.loc[por_mes['ventas_total'].idxmin(), 'mes_nombre']

print(f"\n  Mejor mes historico:  {mejor_mes}")
print(f"  Mes mas tranquilo:    {peor_mes}")

# ── RESUMEN EJECUTIVO ────────────────────────────────
print(f"\n\nCONCLUSIONES PARA LA VICTORIA:")
print(f"  {'─'*60}")
print(f"  1. Programar pedidos a proveedores los dias previos")
print(f"     al mejor dia de ventas para garantizar stock.")
print(f"\n  2. La quincena con mas ventas define cuando")
print(f"     el cliente tiene mas liquidez para comprar.")
print(f"\n  3. Los meses de mayor venta historica son los")
print(f"     momentos para ejecutar promociones y ofertas.")
print(f"\n  4. Los meses tranquilos son para ordenar inventario,")
print(f"     negociar con proveedores y capacitarse.")

ANALISIS DE DIAS Y MESES DE MAYOR VENTA

VENTAS POR DIA DE LA SEMANA:
  Dia            Ventas Total   Facturas   Ticket Prom   Intensidad
  -----------------------------------------------------------------
  Lunes            26,395,430      2,851         9,258   ||||||||||||||
  Martes           33,528,212      2,795        11,996   ||||||||||||||||||
  Miercoles        32,654,974      2,645        12,346   ||||||||||||||||||
  Jueves           31,429,265      2,663        11,802   |||||||||||||||||
  Viernes          32,110,269      2,766        11,609   |||||||||||||||||
  Sabado           36,100,769      3,445        10,479   ||||||||||||||||||||

  Mejor dia de ventas:  Sabado
  Dia mas tranquilo:    Lunes


VENTAS POR QUINCENA DEL MES:
  Quincena                       Ventas Total   Facturas   Ticket Prom
  --------------------------------------------------------------------
  Primera quincena (1-15)          93,539,145      8,271        11,309
  Segunda quincena (16-31)         9

In [54]:
# ══════════════════════════ — PROVEEDORES CON CODIGOS EXACTOS
# Usando codigo de proveedor para evitar
# coincidencias incorrectas
# ══════════════════════════════════════════════════

# Codigos exactos confirmados
codigos_estrategicos = {
    'NAVEMAR'              : None,   # buscar por nombre
    'DISTRIBUIDORA XANDER' : 4,
    'GRUPO S Y S'          : None,
    'REPUESTOS MUNDIALES'  : None,
    'KINSEI'               : 223,
    'AUTO REPUESTOS AV DIEZ': 3,
    'GIGANTE'              : None,
}

print("PROVEEDORES ESTRATEGICOS — VERIFICACION FINAL")
print("=" * 60)

proveedores_validados = []

for nombre_key, codigo in codigos_estrategicos.items():

    if codigo:
        # Buscar por codigo exacto
        coincidencia = df_proveedores[
            df_proveedores['Código'] == codigo
        ]
    else:
        # Buscar por nombre
        coincidencia = df_proveedores[
            df_proveedores['Nombre comercial'].str.contains(
                nombre_key, na=False
            )
        ]

    if len(coincidencia) > 0:
        row = coincidencia.iloc[0]
        proveedores_validados.append({
            'key'    : nombre_key,
            'codigo' : row['Código'],
            'nombre' : row['Nombre comercial'],
        })
        print(f"  [{nombre_key}]")
        print(f"   Nombre oficial: {row['Nombre comercial']}")
        print(f"   Codigo:         {row['Código']}")
        print()
    else:
        print(f"  [{nombre_key}] — No encontrado, confirmar nombre")
        print()

print(f"Total proveedores estrategicos validados: {len(proveedores_validados)}")

PROVEEDORES ESTRATEGICOS — VERIFICACION FINAL
  [NAVEMAR]
   Nombre oficial: NAVEMAR SOCIEDAD ANONIMA
   Codigo:         23

  [DISTRIBUIDORA XANDER]
   Nombre oficial: DISTRIBUIDORA XANDER DCRX SOCIEDAD ANONIMA
   Codigo:         4

  [GRUPO S Y S]
   Nombre oficial: GRUPO S Y S DIVISION AUTOMOTRIZ SOCIEDAD ANONIMA
   Codigo:         57

  [REPUESTOS MUNDIALES]
   Nombre oficial: REPUESTOS MUNDIALES SOCIEDAD ANONIMA
   Codigo:         29

  [KINSEI]
   Nombre oficial: KINSEI LIMITADA
   Codigo:         223

  [AUTO REPUESTOS AV DIEZ]
   Nombre oficial: AUTO REPUESTOS AV DIEZ SOCIEDAD ANONIMA
   Codigo:         3

  [GIGANTE]
   Nombre oficial: REPUESTOS GIGANTE SOCIEDAD ANONIMA
   Codigo:         15

Total proveedores estrategicos validados: 7


In [55]:
# ═════════════════════════════ — ANALISIS COMPLETO
# PROVEEDORES ESTRATEGICOS
# Version autónoma — no depende de celdas anteriores
# ══════════════════════════════════════════════════

import pandas as pd

print("ANALISIS DE PROVEEDORES ESTRATEGICOS")
print("Total Automotriz La Victoria")
print("=" * 70)

# Proveedores validados con sus nombres exactos del catalogo
proveedores_estrategicos = [
    'NAVEMAR',
    'DISTRIBUIDORA XANDER DCRX',
    'GRUPO S Y S',
    'REPUESTOS MUNDIALES',
    'KINSEI LIMITADA',
    'AUTO REPUESTOS AV DIEZ',
    'GIGANTE',
]

# Verificar columnas disponibles en articulos
print(f"\nColumnas en df_articulos:")
print(df_articulos.columns.tolist())

# Identificar columna de proveedor
col_prov = None
for col in df_articulos.columns:
    if 'PROVEEDOR' in str(col).upper() or 'PROV' in str(col).upper():
        col_prov = col
        print(f"\nColumna de proveedor encontrada: '{col_prov}'")
        break

if col_prov is None:
    print("\nNo hay columna de proveedor en articulos.")
    print("Analizando desde el catalogo de proveedores directamente.")

    # Mostrar los proveedores estrategicos del catalogo
    print(f"\n{'='*70}")
    print(f"PROVEEDORES ESTRATEGICOS — INFORMACION DEL CATALOGO")
    print(f"{'='*70}")

    for prov in proveedores_estrategicos:
        coincidencia = df_proveedores[
            df_proveedores['Nombre comercial'].str.upper().str.contains(
                prov.split()[0], na=False
            )
        ]
        if len(coincidencia) > 0:
            row = coincidencia.iloc[0]
            print(f"\n  {prov}")
            print(f"  {'─'*55}")
            for col in df_proveedores.columns:
                val = row[col]
                if pd.notna(val) and str(val).strip() not in ['', 'NAN', 'NONE']:
                    print(f"  {str(col):<25} {str(val)}")
        else:
            print(f"\n  {prov} — No encontrado en catalogo")

else:
    # Si hay columna de proveedor en articulos
    df_articulos[col_prov] = (
        df_articulos[col_prov].astype(str).str.upper().str.strip()
    )

    resumen = []

    for prov in proveedores_estrategicos:
        palabra_clave = prov.split()[0]

        df_prov = df_articulos[
            df_articulos[col_prov].str.contains(palabra_clave, na=False)
        ].copy()

        if len(df_prov) == 0:
            print(f"\n  {prov} — Sin articulos en catalogo")
            continue

        total       = len(df_prov)
        con_stock   = len(df_prov[df_prov['Inventario'] > 0])
        sin_stock   = len(df_prov[df_prov['Inventario'] == 0])
        margen_prom = df_prov['%Utilidad'].mean()
        precio_prom = df_prov['Precio venta'].mean()
        categorias  = df_prov['Línea'].nunique()
        valor_inv   = (df_prov['Inventario'] * df_prov['Costo neto']).sum()

        resumen.append({
            'Proveedor'  : prov,
            'Articulos'  : total,
            'Con stock'  : con_stock,
            'Sin stock'  : sin_stock,
            'Margen'     : margen_prom,
            'Precio prom': precio_prom,
            'Categorias' : categorias,
            'Valor inv'  : valor_inv,
        })

        print(f"\n{'='*70}")
        print(f"  {prov}")
        print(f"{'='*70}")
        print(f"  Articulos en catalogo:       {total:>8,}")
        print(f"  Con stock actual:            {con_stock:>8,}")
        print(f"  Sin stock (agotados):        {sin_stock:>8,}")
        print(f"  Categorias que surte:        {categorias:>8,}")
        print(f"  Margen promedio:             {margen_prom:>7.1f}%")
        print(f"  Precio promedio:             {precio_prom:>10,.0f}")
        print(f"  Valor en inventario:         {valor_inv:>10,.0f}")

        # Top 5 categorias
        cats = df_prov['Línea'].value_counts().head(5)
        print(f"\n  Categorias principales:")
        for cat, cnt in cats.items():
            print(f"    {cat:<35} {cnt:>4} articulos")

        # Top 5 por margen
        top5 = df_prov.nlargest(5, '%Utilidad')[
            ['Nombre','Precio venta','%Utilidad','Inventario']
        ]
        print(f"\n  Top 5 articulos por margen:")
        print(f"  {'Articulo':<48} {'Precio':>10} {'Margen':>8} {'Stock':>7}")
        print(f"  {'─'*76}")
        for _, row in top5.iterrows():
            nombre    = str(row['Nombre'])[:46]
            stock_txt = 'AGOTADO' if row['Inventario'] == 0 \
                        else str(int(row['Inventario']))
            print(f"  {nombre:<48} "
                  f"  {row['Precio venta']:>8,.0f} "
                  f"  {row['%Utilidad']:>6.1f}% "
                  f"  {stock_txt:>7}")

    # Ranking final si hay datos
    if resumen:
        df_res = pd.DataFrame(resumen)

        df_res['score'] = (
            (df_res['Articulos'] / df_res['Articulos'].max() * 0.35) +
            (df_res['Margen']    / df_res['Margen'].max()    * 0.40) +
            (df_res['Con stock'] / df_res['Con stock'].max() * 0.25)
        ).round(3)

        df_res = df_res.sort_values('score', ascending=False).reset_index(drop=True)

        print(f"\n\n{'='*70}")
        print(f"  RANKING FINAL — CON QUIEN EMPEZAR A COMPRAR")
        print(f"  Peso: Margen 40% | Articulos 35% | Stock disponible 25%")
        print(f"{'='*70}")
        print(f"\n  {'#':<4} {'Proveedor':<28} {'Arts':>6} {'Margen':>8} "
              f"{'Categ':>7} {'Score':>8}")
        print(f"  {'─'*65}")

        for i, row in df_res.iterrows():
            print(f"  {i+1:<4} {row['Proveedor']:<28} "
                  f"{int(row['Articulos']):>6,} "
                  f"{row['Margen']:>7.1f}% "
                  f"{int(row['Categorias']):>7} "
                  f"{row['score']:>8.3f}")

        print(f"\n  RECOMENDACION:")
        print(f"  Iniciar negociacion con los 3 primeros del ranking.")
        print(f"  Los 4 restantes se activan en las primeras 2 semanas.")

ANALISIS DE PROVEEDORES ESTRATEGICOS
Total Automotriz La Victoria

Columnas en df_articulos:
['Código', 'Cód. predeterminado', 'Nombre', 'Inv.Stock', 'Inventario', 'Precio venta', 'Estado', 'Cabys', 'Línea', 'Costo neto', '%Utilidad', '%Impuesto', 'Moneda', 'Ubicación', 'nivel_stock']

No hay columna de proveedor en articulos.
Analizando desde el catalogo de proveedores directamente.

PROVEEDORES ESTRATEGICOS — INFORMACION DEL CATALOGO

  NAVEMAR
  ───────────────────────────────────────────────────────
  Código                    23
  Nombre comercial          NAVEMAR SOCIEDAD ANONIMA
  Nombre fiscal             NAVEMAR SOCIEDAD ANONIMA
  Número identificación     3101035984
  Teléfonos                 2283-2430
  Estado                    Activo
  Límite crédito            200000.0
  Plazo crédito             30.0
  Fecha                     2020-03-12 00:00:00
  Facturación               Crédito

  DISTRIBUIDORA XANDER DCRX
  ───────────────────────────────────────────────────────
 